<a href="https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

 1. This notebook designed to show you the bridge between human intuition and machine learning. Think of it as a competition between a rule you create yourself and a rule the computer 'learns' from data.


Here is the breakdown of what is happening:

   *  The Goal: We want to predict which web pages are 'declining' (losing traffic) so we can fix them.
   * The Hand Rule: In the section titled 'A rule you write by hand', the code uses a simple logic: if a page is old (stale) and people still see it (visible), it's important. It calculates a score based on that.
   * The Decision Tree: In section 2, instead of us telling the computer the rule, we give it features (like word_count or avg_position) and ask it to find the best way to split the data to find declining pages. The 'tree' printout is just a series of if/else statements the model discovered.
   * Leakage: Section 3 warns you about 'cheating.' If you give the model a feature that already contains the answer (like the exact percentage of the trend), the model will look perfect, but it won't be useful for future predictions

## 0. Setup (Colab or local)

1. The Environment (The first half)

    import statements: It brings in tools like pandas (for data tables) and os/sys (to talk to the computer).
    git clone & pip install: If you are in Colab, it downloads the necessary files and extra 'toolkits' (libraries) needed for the project.

2. The Data (The second half)

    pd.read_csv(...): This loads a table of 30,000 web pages into a variable called df (short for DataFrame).
    Creating the Label: Look at the line df["is_declining_label"] = .... This is the most important part! It creates a column of 0s and 1s

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)



    1. .str.lower(): It makes sure the text is lowercase so it doesn't get confused by 'Down' vs 'down'.
    2. .eq("down"): This is the 'logic gate.' It asks: 'Does this column exactly equal the word "down"?'
    The Result: This creates a list of True (if it is 'down') or False (if it is 'up' or 'stable').
    3. .astype(int): Computers like numbers better than words. This turns every True into a 1 and every False into a 0.

So, in this notebook, a '1' (the target we want to predict) literally just means 'the trend direction was down'.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

This cell is where you play the role of the 'brain' instead of the model! You are creating a Hand Rule based on intuition. Let's break down the logic of those three lines:

   1.  stale & visible: These are your filters. You've decided a page is only worth looking at if it hasn't been updated in 6 months (>= 180 days) and it still gets at least 500 views (impressions). Just like before, these become 1s (Yes) or 0s (No).

   2.  The Score Calculation: Look at stale * visible * df["impressions_90d"].
        If a page is not stale (0) OR not visible (0), what happens to the final score when you multiply everything by that zero?
        If both are true (1 * 1), the score just becomes the number of impressions.

    Ranking: sort_values(..., ascending=False) puts the highest scores at the top so you can see the 'most important' declining pages first.


In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    #sorts your pages from highest score to lowest. It’s like putting the 'most likely' pages at the front of the line.
    topk = np.asarray(labels)[order[:k]]
    #This looks at only the first K pages (e.g., the first 20 or 50) and pulls their real labels (the 1s and 0s we talked about earlier).
    return topk.mean()
  #.mean(): Since 1 is 'Yes' and 0 is 'No', the average of that list tells you the percentage of 'Yes' hits.
y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
            # You are choosing which columns the computer is allowed to look at (like word count or CTR).
            #Notice it is not allowed to look at the trend direction yet!
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
# Real-world data often has missing values (NaN) or infinity errors. This line cleans the data so the math doesn't break.
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
#You can only ask two levels of questions.
tree.fit(X, y)
# This is where the 'learning' happens. The computer tries thousands of different 'If/Else'
#combinations to see which ones best separate the declining pages from the steady ones.
print(export_text(tree, feature_names=features))
#: This prints out the final logic the computer settled on

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



xactly! The questions change because the computer is trying to be as specific as possible. Think of it like playing a game of 20 Questions:

    Round 1: You ask, 'Is it an animal?'
    If YES: Your next question might be 'Does it have fur?'
    If NO: Your next question would be completely different, like 'Is it made of metal?'

The Decision Tree does the same thing. In the code cell where you trained the model, the tree found that if a page has very few views (impressions <= 5.5), the most important next question is about its position. But if it has many views (impressions > 5.5), it switches to asking about its age (content_age_days).

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

Think of it this way:

    The Hand Rule is like a 'sniper.' It is very good at finding the most obvious, extreme cases at the very top of the list.
    The Decision Tree is more like a 'net.' It might not be as sharp at the very top, but it often stays more consistent as the list gets longer.

There is also a hidden reason why the tree's score looks lower here. In the cell where you printed the tree logic, it only has 4 possible outcomes (the 'leaves'). This means for the top 50 pages, many of them have the exact same score, making it hard for the model to rank one above the other.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

Data leakage is when your model accidentally gets a 'sneak peek' at the answer key during training, which makes it look perfect in class but fail in the real world.

In [ ]:
# 1. We add 'trend_pct' to the features. WARNING: This is the 'answer key'!
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)

# 2. Train the model on this 'leaky' data
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)

# 3. This score looks perfect (1.000) because the model is just looking at the answer
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")

# 4. Look at the logic: notice how it ignores other features and only uses trend_pct
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



### 📝 Key Lesson: What is Data Leakage?

**In simple terms:** Data leakage is when you accidentally include the 'answer' in your training data.

*   **The Cheat:** In the code above, we added `trend_pct` as a feature. Since our goal was to predict if a trend was 'down', and `trend_pct` literally calculates that trend, the model didn't 'learn' anything—it just read the answer.
*   **The Danger:** A leaky model will have a perfect score during testing (like the 1.000 Precision above) but will be **useless** in the real world because you won't have the 'answer' yet when you are trying to make a future prediction.

**Rule of Thumb:** If a piece of information wouldn't be known in the real world at the exact moment you need to make a prediction, don't let your model see it!

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
# Your experiment here
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
            # You are choosing which columns the computer is allowed to look at (like word count or CTR).
            #Notice it is not allowed to look at the trend direction yet!
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
# Real-world data often has missing values (NaN) or infinity errors. This line cleans the data so the math doesn't break.
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
#You can only ask two levels of questions.
tree.fit(X, y)
# This is where the 'learning' happens. The computer tries thousands of different 'If/Else'
#combinations to see which ones best separate the declining pages from the steady ones.
print(export_text(tree, feature_names=features))
#: This prints out the final logic the computer settled on

tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- content_age_days <= 237.50
|   |   |   |   |--- class: 0
|   |   |   |--- content_age_days >  237.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 14.00
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  14.00
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  2.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- impressions_90d <= 73.50
|   |   |   |   |--- class: 1
|   |   |  

In [ ]:
# Your experiment here
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]
            # You are choosing which columns the computer is allowed to look at (like word count or CTR).
            #Notice it is not allowed to look at the trend direction yet!
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
# Real-world data often has missing values (NaN) or infinity errors. This line cleans the data so the math doesn't break.
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
#You can only ask two levels of questions.
tree.fit(X, y)
# This is where the 'learning' happens. The computer tries thousands of different 'If/Else'
#combinations to see which ones best separate the declining pages from the steady ones.
print(export_text(tree, feature_names=features))
#: This prints out the final logic the computer settled on

tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- engagement_rate <= 50.00
|   |   |   |   |--- class: 0
|   |   |   |--- engagement_rate >  50.00
|   |   |   |   |--- class: 1
|   |   |--- word_count >  669.50
|   |   |   |--- days_since_last_update <= 3.50
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  3.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- word_count <= 1653.50
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  1653.50
|   |   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- content_age_days <= 172.50
|   |   |   |   |--- class: 1
|   |   |   |--- content_age_days >  172.50
|   |   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- ctr <= 4.52
|  

### Final Experiment Conclusions

* **Depth 4:** Increasing depth to 4 made the tree harder to read without providing a significant performance boost, suggesting overfitting.
* **Feature Swapping:** Replacing `impressions_90d` with `engagement_rate` was more effective than increasing depth. This change allowed the Decision Tree to outperform the Hand Rule at Precision@50 (0.720 vs 0.680).
* **In-Sample Caveat:** Since these results are in-sample, the next critical step is to validate if these improvements hold up on a client-holdout split to ensure the model isn't just memorizing specific data points.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.